# Week 4 Project - Titanic End-to-End
Full EDA + Feature Engineering on Titanic

# DS imports

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns

## Importing Titanic - and droppping unnecessary or unusable features.

In [3]:
df = pd.read_csv('Titanic-Dataset.csv')
df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

## Sciki-learn Imports

In [22]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.svm import LinearSVC, SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.compose import make_column_transformer, ColumnTransformer
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier


### Preprocessing

In [5]:
ohe = OneHotEncoder(sparse_output=False)
scaler = StandardScaler()
imp_mode = SimpleImputer(strategy='most_frequent')
imp_median = SimpleImputer(strategy='median')

df_e = df[['Embarked']]

df_e = imp_mode.fit_transform(df_e) # Impute The Embarked Column
df['Embarked'] = pd.DataFrame(df_e, columns=['Embarked']) # Insert it back to the DF.

In [6]:
ct = make_column_transformer(
    (scaler, ['Fare']),
    (imp_median, ['Age']),
    (ohe, ['Sex', 'Embarked']),
    remainder='passthrough'
)
ct.set_output(transform='pandas')
df_p = ct.fit_transform(df)


In [7]:
df_p.columns
X = df_p.drop(columns=['remainder__Survived'])
y = df_p[['remainder__Survived']]

In [ ]:
# Grid Search to Determine Best Params - 8 mins to run. 
param_grid_dt = {
    'max_depth': [3, 5, 10, 15, 20], 
    'criterion': ['gini', 'entropy'],
    'min_samples_split': [2, 3, 5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 5],
}

param_grid_rfc = {
    'n_estimators': [50, 100, 150, 200],
    'criterion': ['gini', 'entropy'],
    'min_samples_split': [2, 3, 5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 5]
}

gridt = GridSearchCV(estimator=DecisionTreeClassifier(), param_grid=param_grid_dt)
gridrfc = GridSearchCV(estimator=RandomForestClassifier(), param_grid=param_grid_rfc)
# gridt.fit(X, y)
# gridrfc.fit(X, y)
print(f"Decision Tree High Score: {gridt.best_score_:.3f}")
print(f"Decision Tree Best params: {gridt.best_params_}")
print(f"Random Forest High Score: {gridrfc.best_score_:.3f}")
print(f"Decision Tree Best params: {gridrfc.best_params_}")
print(f"RFC Scores: {gridrfc.cv_results_}")

In [ ]:
# Randomxied for GradientBoosting - Doesn't coverge with GridSearch
grid = RandomizedSearchCV(
    estimator=GradientBoostingClassifier(),
    param_distributions={
        'loss': ['log_loss', 'exponential'], 
        'learning_rate': [0.1, 0.3, 0.5], 
        'n_estimators': [10, 30, 50, 100], 
        'min_samples_split': [2, 3, 5, 10, 15], 
        'min_samples_leaf': [1, 3, 5, 10], 
        'max_depth': [3, 5, 7, 10],
    },
    n_jobs=-1, 
    error_score='raise'
)

# grid.fit(X, y)
print(f"Best Parameters: {grid.best_params_}")
print(f"Best Score: {grid.best_score_}")

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Best from GridSearch. 
dt = DecisionTreeClassifier(criterion='entropy', max_depth=10, min_samples_leaf=2, min_samples_split=15, random_state=42) 
rfc = RandomForestClassifier(n_estimators=50, criterion='entropy', min_samples_leaf=1, min_samples_split=5, oob_score=True, random_state=42) 
gbc = GradientBoostingClassifier(loss='exponential', n_estimators=30, min_samples_split=10,  min_samples_leaf=10, max_depth=10)
dt.fit(X_train, y_train)
rfc.fit(X_train, y_train) # rfc wins
gbc.fit(X_train, y_train)
rfc.score(X_test, y_test)

c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


0.8379888268156425

In [ ]:
DecisionTreeRegressor()